# H4 — Target amino-acid sequence similarity

## Hypothesis
Adverse drug pairs act on proteins that are more evolutionarily/structurally similar (by amino-acid sequence) than non-interacting pairs, **even when the exact targets differ**. This complements H1 (biological overlap, which measures *identity* overlap of target/pathway/GO sets) by measuring *continuous* protein-protein similarity instead of exact-match overlap.

*Interpretation*: "Are the drugs acting on proteins that are evolutionarily or structurally similar, even if they are not the exact same proteins?"

*Caveat*: two homologous proteins may still have quite different ligand specificity or physiology -- high sequence similarity is a necessary-feeling but not sufficient signal for a shared pharmacological mechanism.

## Pipeline
1. Pool every unique target amino-acid sequence (`target_fasta_sequences_1`/`_2`, already parsed per-drug in `data_curation_notebook_v2.ipynb`) across both pair populations into one protein registry.
2. Define **three** candidate protein-protein similarity kernels over that registry:
   - **Kernel A -- hashed k-mer fingerprint + Tanimoto** (alignment-free, same hash-a-local-pattern-into-a-bit idea as the Morgan fingerprint in `h3_structural_similarity`, scales to the full registry).
   - **Kernel B -- k-mer composition cosine similarity** (alignment-free, captures amino-acid composition rather than just presence/absence, also scales to the full registry).
   - **Kernel C -- global (Needleman-Wunsch) alignment percent identity** (the biologically standard notion of sequence similarity, but O(len_a * len_b) per pair -- too slow for the full ~2,200-protein registry, so only computed on a sampled subset of drug pairs for comparison).
3. Define a **set-aggregation** function that turns a protein x protein similarity matrix into a single per-drug-pair score, given each drug's target protein set: `mean_similarity`, `max_similarity`, and `best_match_avg` (a symmetric, continuous analogue of reciprocal-best-hit ortholog matching).
4. Apply kernels A & B to the **full** adverse/non-interacting datasets (471K / 163K pairs).
5. Apply kernel C to a **sampled** subset of pairs only.
6. Compare all three kernels' ability to separate adverse vs. non-interacting pairs on that sample, and pick which kernel(s) to carry forward.
7. Save the full-scale results as `adverse_amino_acid_sim.parquet` / `non_interacting_amino_acid_sim.parquet`, matching the pair-list schema used by `h1_biological_overlap` / `h3_structural_similarity`.


In [1]:
# Imports
import sys
import time
import zlib
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import roc_auc_score
from scipy.stats import mannwhitneyu, spearmanr
from Bio.Align import PairwiseAligner, substitution_matrices

H4_DIR = Path(r"C:\Users\ashto\ddi-prediction\notebooks\h4_target_amino_acid_sim")


## 1. Load the pair data

Same source tables `h1_biological_overlap`/`h3_structural_similarity` use -- direct outputs of `data_curation_notebook_v2.ipynb` (Section 11 saves `adverse_final_df.parquet` / `negative_final_df.parquet` to `data/sample/`). Each row already carries `target_fasta_sequences_1`/`_2` -- one amino-acid sequence per resolved target, aligned with `target_uniprot_ids_1`/`_2`.


In [2]:
adverse_pairs_df = pd.read_parquet(r"C:\Users\ashto\ddi-prediction\data\sample\adverse_final_df.parquet")
non_interacting_pairs_df = pd.read_parquet(r"C:\Users\ashto\ddi-prediction\data\sample\negative_final_df.parquet")

print(f"adverse:         {adverse_pairs_df.shape}")
print(f"non_interacting: {non_interacting_pairs_df.shape}")
adverse_pairs_df[["drug1_id", "drug2_id", "target_uniprot_ids_1", "target_fasta_sequences_1"]].head(3)


adverse:         (471169, 17)
non_interacting: (162788, 15)


,drug1_id,drug2_id,target_uniprot_ids_1,target_fasta_sequences_1
0,DB00006,DB06605,[P00734],[MAHVRGLQLPGCLALAALCSLVHSQHVFLAPQQARSLLQRVRRAN...
1,DB00006,DB06695,[P00734],[MAHVRGLQLPGCLALAALCSLVHSQHVFLAPQQARSLLQRVRRAN...
2,DB00006,DB01254,[P00734],[MAHVRGLQLPGCLALAALCSLVHSQHVFLAPQQARSLLQRVRRAN...


## 2. Build the protein registry

Pool every unique target sequence across *both* pair populations into one registry, deduplicated on the sequence itself (not the UniProt id -- isoforms/near-duplicate accessions with an identical sequence should collapse to one kernel row). Every kernel below is computed once over this registry; per-pair scores are then looked up by index instead of recomputing anything per row.


In [3]:
def collect_unique_sequences(*dfs, seq_cols=("target_fasta_sequences_1", "target_fasta_sequences_2")):
    """Pool every non-empty amino-acid sequence across `dfs` into one sorted, deduplicated list."""
    seqs = set()
    for df in dfs:
        for col in seq_cols:
            for arr in df[col]:
                if arr is None:
                    continue
                seqs.update(s for s in arr if s)
    return sorted(seqs)


protein_sequences = collect_unique_sequences(adverse_pairs_df, non_interacting_pairs_df)
protein_index = {seq: i for i, seq in enumerate(protein_sequences)}
n_proteins = len(protein_sequences)

seq_lengths = np.array([len(s) for s in protein_sequences])
print(f"Unique target sequences pooled across both pair sets: {n_proteins:,}")
print(f"Sequence length: min={seq_lengths.min()}, median={int(np.median(seq_lengths))}, "
      f"max={seq_lengths.max()}, mean={seq_lengths.mean():.0f}")


Unique target sequences pooled across both pair sets: 2,221
Sequence length: min=11, median=469, max=14507, mean=608


## 3. Kernel A — hashed k-mer fingerprint + Tanimoto similarity

Same idea as the Morgan fingerprint in `h3_structural_similarity`: hash every local pattern (there, a circular atom neighborhood; here, an overlapping amino-acid k-mer) into one bit of a fixed-width vector, then compare two proteins with Tanimoto similarity. Alignment-free and $O(n)$ per sequence, so it scales to the full protein registry (and, later, to the full 634K-pair dataset). We use `zlib.crc32` (not Python's built-in `hash()`) because string hashing in Python is randomized per-process by default -- an unstable hash would make fingerprints (and therefore parquet outputs) non-reproducible across kernel restarts.


In [4]:
AA_KMER_K = 3
AA_FP_N_BITS = 2048


def sequence_to_kmer_bits(seq, k=AA_KMER_K, n_bits=AA_FP_N_BITS):
    """Hash every overlapping k-mer of an amino-acid sequence into a fixed-width bit vector."""
    bits = np.zeros(n_bits, dtype=bool)
    if not seq or len(seq) < k:
        return bits
    for i in range(len(seq) - k + 1):
        idx = zlib.crc32(seq[i:i + k].encode("ascii")) % n_bits
        bits[idx] = True
    return bits


def build_kmer_fingerprint_matrix(sequences, k=AA_KMER_K, n_bits=AA_FP_N_BITS):
    """One boolean fingerprint row per protein sequence, in registry order."""
    fp = np.zeros((len(sequences), n_bits), dtype=bool)
    for i, seq in enumerate(sequences):
        fp[i] = sequence_to_kmer_bits(seq, k=k, n_bits=n_bits)
    return fp


def tanimoto_matrix(bit_matrix):
    """Vectorized pairwise Tanimoto similarity for an (n_proteins, n_bits) boolean matrix."""
    bits = bit_matrix.astype(np.float32)
    intersection = bits @ bits.T
    on_counts = bits.sum(axis=1)
    union = on_counts[:, None] + on_counts[None, :] - intersection
    with np.errstate(divide="ignore", invalid="ignore"):
        sim = np.where(union > 0, intersection / union, 0.0)
    np.fill_diagonal(sim, 1.0)
    return sim.astype(np.float32)


t0 = time.time()
kmer_fp_matrix = build_kmer_fingerprint_matrix(protein_sequences)
kmer_tanimoto_sim = tanimoto_matrix(kmer_fp_matrix)
print(f"Kernel A (k-mer fingerprint Tanimoto): {kmer_tanimoto_sim.shape} matrix in {time.time() - t0:.1f}s")


Kernel A (k-mer fingerprint Tanimoto): (2221, 2221) matrix in 0.5s


## 4. Kernel B — k-mer composition cosine similarity

Also alignment-free, but keeps the actual tripeptide **counts** instead of collapsing them into a single hashed bit -- captures amino-acid composition (how much of each k-mer a protein has), whereas Kernel A only captures presence/absence of the hashed slots.


In [5]:
def build_kmer_count_matrix(sequences, k=AA_KMER_K):
    """Sparse tripeptide count matrix (n_proteins, n_kmers) via a char-ngram vectorizer."""
    vectorizer = CountVectorizer(analyzer="char", ngram_range=(k, k), lowercase=False)
    return vectorizer.fit_transform(sequences)


t0 = time.time()
kmer_count_matrix = build_kmer_count_matrix(protein_sequences)
kmer_cosine_sim = cosine_similarity(kmer_count_matrix).astype(np.float32)
print(f"Kernel B (k-mer composition cosine): {kmer_cosine_sim.shape} matrix in {time.time() - t0:.1f}s, "
      f"vocab={kmer_count_matrix.shape[1]:,} distinct tripeptides")


Kernel B (k-mer composition cosine): (2221, 2221) matrix in 1.2s, vocab=8,048 distinct tripeptides


## 5. Kernel C — global (Needleman-Wunsch) alignment percent identity

The biologically standard notion of sequence similarity: align two sequences end-to-end with BLOSUM62 substitution scores and affine gap penalties, then normalize by the better sequence's own self-alignment score (a BLOSUM-weighted analogue of percent identity, bounded roughly in $[0, 1]$). This is $O(\text{len}_a \times \text{len}_b)$ per pair -- with ~2,200 registry proteins (up to 14,507 residues each) a full registry matrix is computationally infeasible here, so Kernel C is only evaluated on a **sampled subset of drug pairs** in Section 7, purely to compare against Kernels A/B.


**Non-standard residue handling:** BLOSUM62's alphabet (`ARNDCQEGHILKMFPSTWYVBZX*`) does not include `U` (selenocysteine) -- confirmed present in this dataset's FASTA sequences by scanning the full registry for out-of-alphabet characters. `_sanitize_for_aligner` remaps any such residue to `X` (BLOSUM62's "unknown" symbol, scored near-neutrally against everything) before alignment, so those handful of selenoprotein residues don't crash the aligner; this only affects Kernel C -- Kernels A/B hash/count raw characters and don't require a fixed alphabet.

In [10]:
_aligner = PairwiseAligner()
_aligner.substitution_matrix = substitution_matrices.load("BLOSUM62")
_aligner.mode = "global"
_aligner.open_gap_score = -10
_aligner.extend_gap_score = -0.5
_aligner_alphabet = set(_aligner.substitution_matrix.alphabet)


def _sanitize_for_aligner(seq, alphabet=_aligner_alphabet):
    """Map any residue outside the substitution matrix's alphabet (e.g. 'U' selenocysteine) to 'X'."""
    return "".join(c if c in alphabet else "X" for c in seq)


def alignment_percent_identity(seq_a, seq_b, aligner=_aligner):
    """Needleman-Wunsch global alignment score, normalized by the better sequence's own
    self-alignment score -- a BLOSUM62-weighted analogue of percent identity."""
    if not seq_a or not seq_b:
        return 0.0
    seq_a = _sanitize_for_aligner(seq_a)
    seq_b = _sanitize_for_aligner(seq_b)
    score_ab = aligner.score(seq_a, seq_b)
    denom = min(aligner.score(seq_a, seq_a), aligner.score(seq_b, seq_b))
    return float(score_ab / denom) if denom > 0 else 0.0


# Smoke test: identical sequences should score ~1.0, unrelated ones much lower.
_sample_a, _sample_b = protein_sequences[0], protein_sequences[1]
print(f"self-identity check: {alignment_percent_identity(_sample_a, _sample_a):.3f} (expect 1.0)")
print(f"sample pair identity: {alignment_percent_identity(_sample_a, _sample_b):.3f}")


self-identity check: 1.000 (expect 1.0)
sample pair identity: -0.044


## 6. Set aggregation — from protein pairs to a drug pair

A drug's "target set" is a handful of proteins, not one; the kernel above only scores *protein pairs*. To score a *drug pair* we aggregate the protein-protein similarity matrix over each drug's target set, `A` (drug 1's targets) and `B` (drug 2's targets):

- **`mean_similarity`**: average similarity over every cross pair in $A \times B$ -- "on average, how similar are the two drugs' targets?"
- **`max_similarity`**: the single best-matching protein pair -- "is there *any* pair of targets that's highly similar?" (sensitive to one lucky/unlucky pair, and to target-set size: bigger sets have more chances at a high max).
- **`best_match_avg`**: symmetric average of each protein's *best* hit in the other drug's set, i.e. $\frac{1}{2}\left(\text{mean}_{i \in A} \max_{j \in B} S_{ij} + \text{mean}_{j \in B} \max_{i \in A} S_{ij}\right)$ -- a continuous, size-robust analogue of reciprocal-best-hit ortholog matching; each target gets one "vote" regardless of how large the *other* set is.

All three are reported per pair (not just one) since they answer subtly different questions and downstream models can decide which is most predictive.


In [7]:
def aggregate_set_similarity(sim_matrix, idx_a, idx_b):
    """Aggregate a protein-protein similarity matrix across two drugs' target-protein sets.

    `idx_a`/`idx_b` are registry indices (or row/col ids) into `sim_matrix` for drug A's and
    drug B's targets. Returns (mean_similarity, max_similarity, best_match_avg).
    """
    if len(idx_a) == 0 or len(idx_b) == 0:
        return 0.0, 0.0, 0.0
    sub = sim_matrix[np.ix_(idx_a, idx_b)]
    mean_sim = float(sub.mean())
    max_sim = float(sub.max())
    best_match_avg = float(0.5 * (sub.max(axis=1).mean() + sub.max(axis=0).mean()))
    return mean_sim, max_sim, best_match_avg


def sequences_to_indices(seq_array, index_map):
    """Map an array of amino-acid sequences to registry indices, dropping empty/unknown entries."""
    return np.array([index_map[s] for s in seq_array if s], dtype=np.int64)


## 7. Apply Kernels A & B to the full datasets

Both kernels above are alignment-free and were already computed once over the whole registry (Sections 3-4), so scoring all 634K pairs is just index lookups + small submatrix aggregations -- no per-row recomputation of the kernel itself.


In [8]:
def compute_kernel_aggregates(df, sim_matrix, index_map, kernel_name):
    """Row-wise mean/max/best-match aggregation of `sim_matrix` across each pair's two target sets."""
    idx1 = df["target_fasta_sequences_1"].apply(lambda a: sequences_to_indices(a, index_map))
    idx2 = df["target_fasta_sequences_2"].apply(lambda a: sequences_to_indices(a, index_map))

    n = len(df)
    means = np.empty(n, dtype=np.float32)
    maxs = np.empty(n, dtype=np.float32)
    best_match = np.empty(n, dtype=np.float32)
    for i, (a, b) in enumerate(zip(idx1, idx2)):
        means[i], maxs[i], best_match[i] = aggregate_set_similarity(sim_matrix, a, b)

    return pd.DataFrame({
        f"{kernel_name}_mean_similarity": means,
        f"{kernel_name}_max_similarity": maxs,
        f"{kernel_name}_best_match_avg": best_match,
    }, index=df.index)


t0 = time.time()
adverse_kmer_fp_df = compute_kernel_aggregates(adverse_pairs_df, kmer_tanimoto_sim, protein_index, "kmer_fp")
non_interacting_kmer_fp_df = compute_kernel_aggregates(non_interacting_pairs_df, kmer_tanimoto_sim, protein_index, "kmer_fp")
print(f"Kernel A aggregated over both datasets in {time.time() - t0:.1f}s")

t0 = time.time()
adverse_kmer_cosine_df = compute_kernel_aggregates(adverse_pairs_df, kmer_cosine_sim, protein_index, "kmer_cosine")
non_interacting_kmer_cosine_df = compute_kernel_aggregates(non_interacting_pairs_df, kmer_cosine_sim, protein_index, "kmer_cosine")
print(f"Kernel B aggregated over both datasets in {time.time() - t0:.1f}s")

adverse_full_scale_df = pd.concat([adverse_kmer_fp_df, adverse_kmer_cosine_df], axis=1)
non_interacting_full_scale_df = pd.concat([non_interacting_kmer_fp_df, non_interacting_kmer_cosine_df], axis=1)
adverse_full_scale_df.describe()


Kernel A aggregated over both datasets in 24.4s
Kernel B aggregated over both datasets in 24.6s


,kmer_fp_mean_similarity,kmer_fp_max_similarity,kmer_fp_best_match_avg,kmer_cosine_mean_similarity,kmer_cosine_max_similarity,kmer_cosine_best_match_avg
count,471169.000000,471169.000000,471169.000000,471169.000000,471169.000000,471169.000000
mean,0.170492,0.256029,0.207040,0.110308,0.191236,0.145374
std,0.082335,0.207328,0.126228,0.071889,0.212608,0.123896
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.137350,0.168576,0.155332,0.080629,0.106139,0.095069
50%,0.166214,0.207303,0.187054,0.102107,0.140936,0.122342
75%,0.200095,0.269841,0.231994,0.128572,0.186712,0.156028
max,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


## 8. Apply Kernel C (alignment) to a sampled subset

Sample `SAMPLE_PER_CLASS` pairs from each population (only pairs where both drugs have at least one resolved target). To keep runtime bounded we also cap, *per sampled pair*, how many target sequences per side enter the alignment ($\le$ `MAX_TARGETS_PER_DRUG_FOR_ALIGNMENT`) and skip any sequence longer than `MAX_PROTEIN_LEN_FOR_ALIGNMENT` residues (a handful of registry proteins run into the thousands of residues, which would dominate runtime for a comparison-only kernel).


In [11]:
SAMPLE_PER_CLASS = 800
MAX_TARGETS_PER_DRUG_FOR_ALIGNMENT = 5
MAX_PROTEIN_LEN_FOR_ALIGNMENT = 2000
RNG_SEED = 42

rng = np.random.default_rng(RNG_SEED)


def sample_pairs_with_targets(df, n, rng=rng):
    """Sample up to `n` rows where both sides have at least one resolved target sequence."""
    has_targets = (df["target_fasta_sequences_1"].apply(len) > 0) & (df["target_fasta_sequences_2"].apply(len) > 0)
    eligible = df.index[has_targets]
    sampled_idx = rng.choice(eligible, size=min(n, len(eligible)), replace=False)
    return df.loc[sampled_idx]


def clipped_sequences(seq_array, max_n=MAX_TARGETS_PER_DRUG_FOR_ALIGNMENT, max_len=MAX_PROTEIN_LEN_FOR_ALIGNMENT):
    """First `max_n` non-empty sequences, dropping any longer than `max_len` residues."""
    return [s for s in seq_array[:max_n] if s and len(s) <= max_len]


def aggregate_alignment_similarity(seqs_a, seqs_b):
    """Same mean/max/best_match_avg aggregation as `aggregate_set_similarity`, but computing the
    protein-protein alignment scores on demand instead of looking them up in a precomputed matrix."""
    if not seqs_a or not seqs_b:
        return 0.0, 0.0, 0.0
    sub = np.array([[alignment_percent_identity(a, b) for b in seqs_b] for a in seqs_a], dtype=np.float32)
    mean_sim = float(sub.mean())
    max_sim = float(sub.max())
    best_match_avg = float(0.5 * (sub.max(axis=1).mean() + sub.max(axis=0).mean()))
    return mean_sim, max_sim, best_match_avg


def compute_alignment_aggregates(df, label, log_every=200):
    rows = []
    t0 = time.time()
    for i, (_, row) in enumerate(df.iterrows()):
        seqs_a = clipped_sequences(row["target_fasta_sequences_1"])
        seqs_b = clipped_sequences(row["target_fasta_sequences_2"])
        rows.append(aggregate_alignment_similarity(seqs_a, seqs_b))
        if (i + 1) % log_every == 0:
            print(f"  [{label}] {i + 1}/{len(df)} pairs aligned ({time.time() - t0:.0f}s elapsed)")
    out = pd.DataFrame(
        rows,
        columns=["alignment_mean_similarity", "alignment_max_similarity", "alignment_best_match_avg"],
        index=df.index,
    )
    print(f"[{label}] Aligned {len(df)} pairs in {time.time() - t0:.1f}s")
    return out


adverse_sample_df = sample_pairs_with_targets(adverse_pairs_df, SAMPLE_PER_CLASS)
non_interacting_sample_df = sample_pairs_with_targets(non_interacting_pairs_df, SAMPLE_PER_CLASS)

adverse_alignment_df = compute_alignment_aggregates(adverse_sample_df, "adverse")
non_interacting_alignment_df = compute_alignment_aggregates(non_interacting_sample_df, "non_interacting")


  [adverse] 200/800 pairs aligned (7s elapsed)
  [adverse] 400/800 pairs aligned (15s elapsed)
  [adverse] 600/800 pairs aligned (23s elapsed)
  [adverse] 800/800 pairs aligned (30s elapsed)
[adverse] Aligned 800 pairs in 29.6s
  [non_interacting] 200/800 pairs aligned (5s elapsed)
  [non_interacting] 400/800 pairs aligned (12s elapsed)
  [non_interacting] 600/800 pairs aligned (19s elapsed)
  [non_interacting] 800/800 pairs aligned (24s elapsed)
[non_interacting] Aligned 800 pairs in 23.6s


## 9. Compare the three kernels

On the sampled subset, all three kernels are available side by side. Two comparisons:
1. **Spearman correlation** between kernels (do they broadly agree on which pairs are "similar"?).
2. **Separation power**: AUROC (and Mann-Whitney U) using each kernel's score to distinguish *adverse* (label=1) from *non_interacting* (label=0) pairs -- a proxy for "which kernel brings the most meaning" to the actual hypothesis being tested, not just internal agreement between kernels.


In [12]:
METRIC_SUFFIXES = ("_mean_similarity", "_max_similarity", "_best_match_avg")


def build_comparison_table():
    """Assemble a label + all-three-kernels table restricted to the sampled rows."""
    adverse_cmp = adverse_full_scale_df.loc[adverse_sample_df.index].copy()
    non_cmp = non_interacting_full_scale_df.loc[non_interacting_sample_df.index].copy()
    adverse_cmp = pd.concat([adverse_cmp, adverse_alignment_df], axis=1)
    non_cmp = pd.concat([non_cmp, non_interacting_alignment_df], axis=1)
    adverse_cmp["label"] = 1
    non_cmp["label"] = 0
    return pd.concat([adverse_cmp, non_cmp], ignore_index=True)


comparison_df = build_comparison_table()
print(f"Comparison sample: {len(comparison_df):,} pairs ({comparison_df['label'].value_counts().to_dict()})")

kernels = ["kmer_fp", "kmer_cosine", "alignment"]

# --- 1. Inter-kernel agreement (Spearman rho on mean_similarity) ---
corr_cols = [f"{k}_mean_similarity" for k in kernels]
kernel_correlation_df = comparison_df[corr_cols].corr(method="spearman")
print("\nInter-kernel Spearman correlation (mean_similarity):")
print(kernel_correlation_df.round(3))


Comparison sample: 1,600 pairs ({1: 800, 0: 800})

Inter-kernel Spearman correlation (mean_similarity):
                             kmer_fp_mean_similarity  \
kmer_fp_mean_similarity                        1.000   
kmer_cosine_mean_similarity                    0.886   
alignment_mean_similarity                      0.269   

                             kmer_cosine_mean_similarity  \
kmer_fp_mean_similarity                            0.886   
kmer_cosine_mean_similarity                        1.000   
alignment_mean_similarity                          0.231   

                             alignment_mean_similarity  
kmer_fp_mean_similarity                          0.269  
kmer_cosine_mean_similarity                      0.231  
alignment_mean_similarity                        1.000  


In [13]:
# --- 2. Separation power: does each kernel/metric distinguish adverse from non_interacting? ---
comparison_rows = []
for kernel in kernels:
    for suffix in METRIC_SUFFIXES:
        col = f"{kernel}{suffix}"
        auroc = roc_auc_score(comparison_df["label"], comparison_df[col])
        stat, p_value = mannwhitneyu(
            comparison_df.loc[comparison_df["label"] == 1, col],
            comparison_df.loc[comparison_df["label"] == 0, col],
            alternative="two-sided",
        )
        comparison_rows.append({
            "kernel": kernel,
            "metric": suffix.lstrip("_"),
            "auroc_vs_label": auroc,
            "mannwhitney_p": p_value,
        })

comparison_summary_df = pd.DataFrame(comparison_rows)
comparison_summary_df["separation"] = (comparison_summary_df["auroc_vs_label"] - 0.5).abs()
comparison_summary_df.sort_values("separation", ascending=False).reset_index(drop=True)


,kernel,metric,auroc_vs_label,mannwhitney_p,separation
0,kmer_cosine,max_similarity,0.614317,2.416692e-15,0.114317
1,kmer_cosine,best_match_avg,0.608552,5.548293e-14,0.108552
2,kmer_cosine,mean_similarity,0.587977,1.106500e-09,0.087977
3,kmer_fp,max_similarity,0.586331,2.238713e-09,0.086331
4,kmer_fp,best_match_avg,0.584578,4.688695e-09,0.084578
5,kmer_fp,mean_similarity,0.564213,8.692279e-06,0.064213
6,alignment,max_similarity,0.556591,8.871934e-05,0.056591
7,alignment,best_match_avg,0.540279,5.275384e-03,0.040279
8,alignment,mean_similarity,0.522049,1.267348e-01,0.022049


In [14]:
FULL_SCALE_KERNELS = {"kmer_fp", "kmer_cosine"}  # alignment (Kernel C) doesn't scale to all 634K pairs

best_kernel = comparison_summary_df.groupby("kernel")["separation"].mean().idxmax()
final_kernel = best_kernel if best_kernel in FULL_SCALE_KERNELS else max(
    FULL_SCALE_KERNELS, key=lambda k: comparison_summary_df.loc[comparison_summary_df["kernel"] == k, "separation"].mean()
)

print(f"Kernel with the strongest mean adverse-vs-non_interacting separation on the sample: '{best_kernel}'")
if best_kernel not in FULL_SCALE_KERNELS:
    n_total = len(adverse_pairs_df) + len(non_interacting_pairs_df)
    print(f"'{best_kernel}' isn't full-scale-tractable ({n_total:,} pairs) -- falling back to "
          f"'{final_kernel}' (alignment-free) for the saved, full-dataset outputs below.")
else:
    print(f"Using '{final_kernel}' for the saved, full-dataset outputs below.")


Kernel with the strongest mean adverse-vs-non_interacting separation on the sample: 'kmer_cosine'
Using 'kmer_cosine' for the saved, full-dataset outputs below.


## 10. Assemble & save the full-dataset outputs

Save both kernels' full-scale mean/max/best-match scores (six columns) plus a `similarity_score` alias -- the winning kernel's `best_match_avg` -- so `network_generation.ipynb` can consume this file exactly like `h1_biological_overlap`/`h3_structural_similarity`'s single-scalar edge weight. The sampled alignment comparison table is saved separately since it does not cover the full pair universe.


In [15]:
def build_output_df(pairs_df, kernel_aggregates_df, final_kernel):
    out = pd.DataFrame({
        "drug1_id": pairs_df["drug1_id"].astype(str),
        "drug2_id": pairs_df["drug2_id"].astype(str),
        "drug1_name": pairs_df["drug1_name"].astype(str),
        "drug2_name": pairs_df["drug2_name"].astype(str),
        "n_targets_1": pairs_df["target_fasta_sequences_1"].apply(len),
        "n_targets_2": pairs_df["target_fasta_sequences_2"].apply(len),
    }, index=pairs_df.index)
    out = pd.concat([out, kernel_aggregates_df], axis=1)
    out["similarity_score"] = out[f"{final_kernel}_best_match_avg"]
    return out.reset_index(drop=True)


adverse_amino_acid_sim_df = build_output_df(adverse_pairs_df, adverse_full_scale_df, final_kernel)
non_interacting_amino_acid_sim_df = build_output_df(non_interacting_pairs_df, non_interacting_full_scale_df, final_kernel)

adverse_amino_acid_sim_df.to_parquet(H4_DIR / "adverse_amino_acid_sim.parquet")
non_interacting_amino_acid_sim_df.to_parquet(H4_DIR / "non_interacting_amino_acid_sim.parquet")
comparison_df.to_parquet(H4_DIR / "amino_acid_kernel_comparison_sample.parquet")

print(f"Saved {adverse_amino_acid_sim_df.shape} to adverse_amino_acid_sim.parquet")
print(f"Saved {non_interacting_amino_acid_sim_df.shape} to non_interacting_amino_acid_sim.parquet")
print(f"Saved {comparison_df.shape} to amino_acid_kernel_comparison_sample.parquet (kernel-comparison sample only)")
adverse_amino_acid_sim_df.head()


Saved (471169, 13) to adverse_amino_acid_sim.parquet
Saved (162788, 13) to non_interacting_amino_acid_sim.parquet
Saved (1600, 10) to amino_acid_kernel_comparison_sample.parquet (kernel-comparison sample only)


,drug1_id,drug2_id,drug1_name,drug2_name,n_targets_1,n_targets_2,kmer_fp_mean_similarity,kmer_fp_max_similarity,kmer_fp_best_match_avg,kmer_cosine_mean_similarity,kmer_cosine_max_similarity,kmer_cosine_best_match_avg,similarity_score
0,DB00006,DB06605,Bivalirudin,Apixaban,1,1,0.208388,0.208388,0.208388,0.161692,0.161692,0.161692,0.161692
1,DB00006,DB06695,Bivalirudin,Dabigatran etexilate,1,1,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
2,DB00006,DB01254,Bivalirudin,Dasatinib,1,23,0.198479,0.243194,0.220837,0.108032,0.149992,0.129012,0.129012
3,DB00006,DB01609,Bivalirudin,Deferasirox,1,0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,DB00006,DB01586,Bivalirudin,Ursodeoxycholic acid,1,4,0.158606,0.209190,0.183898,0.088321,0.108338,0.098330,0.098330


## Summary & limitations

**Outputs:**
- `adverse_amino_acid_sim.parquet` / `non_interacting_amino_acid_sim.parquet` -- one row per pair with `n_targets_1/2`, all six alignment-free kernel metrics (`kmer_fp_*`, `kmer_cosine_*`), and `similarity_score` (the winning kernel's `best_match_avg`, used as the graph edge weight in `network_generation.ipynb`).
- `amino_acid_kernel_comparison_sample.parquet` -- the ~1,600-pair sample where all three kernels (including alignment) are available, for reference/re-analysis.

**Known limitations:**
1. Kernel C (global alignment) only ran on a sample, capped to `MAX_TARGETS_PER_DRUG_FOR_ALIGNMENT` targets/side and `MAX_PROTEIN_LEN_FOR_ALIGNMENT` residues/protein -- it is a comparison reference, not part of the saved full-dataset scores.
1a. Kernel C also remaps the rare `U` (selenocysteine) residue to `X` before alignment (see Section 5) since BLOSUM62 has no score for it -- a minor approximation affecting only the small number of selenoprotein sequences in the registry.
2. `best_match_avg` rewards *any* strongly similar protein pair between two drugs' target sets, regardless of whether that similarity reflects a shared pharmacological mechanism -- per the hypothesis caveat, two homologous proteins can still have very different ligand specificity or physiology, so a high score here is necessary-feeling, not sufficient, evidence of mechanistic overlap.
3. `mean_similarity` is sensitive to target-set size mismatches (a promiscuous drug with many weakly-related targets dilutes its own mean), and `max_similarity` is sensitive to target-set size in the opposite direction (bigger sets get more "lottery tickets" at a high max) -- `best_match_avg` is the more size-robust default, but all three are kept as features.
4. Kernels A/B are alignment-free proxies for sequence similarity (hashed/counted k-mers), not the biological gold standard -- Section 9's comparison against Kernel C on the sample is the check that they're not wildly misaligned with true homology signal.


## Conclusions & recommendations (from the executed run)

### Conclusions

1. **`kmer_cosine` (k-mer composition cosine similarity) won the kernel comparison**, beating both `kmer_fp` (hashed fingerprint Tanimoto) and `alignment` (Needleman-Wunsch percent identity) on every aggregation metric:

   | kernel | metric | auroc_vs_label | separation |
   |---|---|---|---|
   | kmer_cosine | max_similarity | 0.614 | 0.114 |
   | kmer_cosine | best_match_avg | 0.609 | 0.109 |
   | kmer_fp | max_similarity | 0.586 | 0.086 |
   | alignment | max_similarity | 0.557 | 0.057 |
   | alignment | mean_similarity | 0.522 | 0.022 (not significant, p=0.13) |

   `final_kernel = 'kmer_cosine'` was selected automatically on this basis, and its `best_match_avg` was saved as `similarity_score` in both output parquet files.
2. **`max_similarity` consistently separates adverse from non-interacting pairs slightly better than `best_match_avg` or `mean_similarity`, for all three kernels.** This suggests the *single most-similar target pair* between two drugs carries more signal than the target sets' overall/average similarity -- one strongly homologous target may matter more than broad but weak target-set relatedness. `best_match_avg` was still used for `similarity_score` since it's the more size-robust default (see Section 6), but `max_similarity` is worth a direct follow-up as its own model feature or edge weight.
3. **The alignment-free kernels (`kmer_fp`, `kmer_cosine`) agree strongly with each other (Spearman rho = 0.89 on `mean_similarity`) but only weakly with the alignment kernel (rho = 0.23-0.27).** Combined with alignment's weaker separation, this is a real divergence, not just sampling noise -- plausible explanations: (a) the alignment kernel's normalization (score / self-alignment score) and per-pair target-count cap (5) reduce its resolution relative to the full k-mer profiles, and (b) k-mer composition may be capturing coarser compositional/physicochemical similarity that happens to track the adverse/non-interacting split better than strict positional homology does at this sample size (n=1,600).
4. **This is a first-pass hypothesis check on a comparison sample, not a validated production kernel.** The 800/800-pair sample is small relative to the full 634K-pair dataset, and the alignment kernel in particular was capped in ways (Section 8) that may understate its true separation power.

### Recommendations

1. **Use `adverse_amino_acid_sim.parquet` / `non_interacting_amino_acid_sim.parquet`'s `similarity_score` (`kmer_cosine_best_match_avg`) as the default H4 feature** for downstream network/embedding work -- proceed with `network_generation.ipynb` next.
2. **Revisit `max_similarity` as an alternate edge weight or standalone feature** in a future ablation, given its slightly stronger separation across all three kernels here.
3. **If alignment-based similarity is worth pursuing further, scale up the sample** (more pairs, more targets/side, no length cap) before concluding it's weaker than the alignment-free kernels -- the current sample is a deliberately cheap comparison, not a fully fair fight.
4. Re-run Section 9's comparison after any future change to `AA_KMER_K`, `AA_FP_N_BITS`, or the alignment gap penalties, since the kernel ranking could shift.
